# 14. SQL RAG - 自然语言转SQL

**复杂度:** ⭐⭐⭐⭐

## 概述

**SQL RAG** 通过结合以下功能实现结构化数据库的自然语言查询:
1. Schema检索(查找相关表)
2. Text-to-SQL生成(LLM生成SQL)
3. SQL执行(在沙箱中安全执行)
4. 结果解释(自然语言回答)

### 问题所在

标准RAG在非结构化文本上表现良好,但在以下场景会失败:
- 结构化数据库(SQL、NoSQL)
- 聚合和分析(COUNT、SUM、AVG)
- 精确数据查找(精确匹配、过滤)
- 关系型查询(跨表JOIN)

**需要SQL的示例查询:**
- "我们在法国有多少客户?"
- "上个月的平均订单价值是多少?"
- "显示按收入排名的前5个产品"
- "哪些员工的销售额超过$50,000?"

### 解决方案

SQL RAG流程:

```
问题 → Schema检索 → Text-to-SQL → SQL验证
    → 执行查询 → 结果 → 自然语言回答
```

### 架构

1. **Schema索引**: 嵌入表/列描述
2. **Schema检索**: 为查询找到相关表
3. **SQL生成**: LLM使用schema上下文生成SQL
4. **安全层**: 验证SQL(只读,防止注入)
5. **执行**: 在受控环境中运行查询
6. **解释**: 将结果转换为自然语言
7. **错误处理**: 如果SQL失败,重试并修正

### 示例数据库: Chinook

我们将使用**Chinook**数据库 - 一个示例音乐商店数据库,包含:
- **Artists**: 乐队/音乐家信息
- **Albums**: 音乐专辑
- **Tracks**: 单独的歌曲
- **Customers**: 客户记录
- **Employees**: 员工信息
- **Invoices**: 销售交易
- **InvoiceLines**: 行项目
- **Playlists**: 歌曲集合

### 何时使用

✅ **适用于:**
- 分析和聚合
- 结构化数据查询
- 企业数据(数据库、数据仓库)
- 精确查找和过滤
- 时间序列数据

❌ **不理想的情况:**
- 非结构化文本文档
- 语义相似度搜索
- 用户不理解数据结构时
- SQL对LLM来说太复杂时

### 权衡

**优点:**
- ✅ 精确答案(无幻觉)
- ✅ 处理聚合和数学运算
- ✅ 与现有数据库配合使用
- ✅ 可验证的结果

**缺点:**
- ❌ 需要良好的schema设计
- ❌ LLM SQL错误很常见
- ❌ 安全考虑
- ❌ 受限于查询表达能力

---

## 实现

## 1. 设置和导入

In [1]:
import sys
import sqlite3
from pathlib import Path
from typing import List, Dict, Any, Tuple
import json

# Add parent directory to path for imports
sys.path.append(str(Path("../..").resolve()))

from shared.config import create_chat_model, create_embeddings, DEFAULT_MODEL, DEFAULT_TEMPERATURE, DEFAULT_VISION_MODEL
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
import pandas as pd

from shared.config import (
    verify_api_key,
    DEFAULT_MODEL,
    DEFAULT_TEMPERATURE,
    HF_EMBEDDING_MODEL,
    VECTOR_STORE_DIR,
    PROJECT_ROOT,
)
from shared.prompts import (
    SQL_SCHEMA_SUMMARY_PROMPT,
    TEXT_TO_SQL_PROMPT,
    SQL_RESULTS_INTERPRETATION_PROMPT,
    SQL_ERROR_RECOVERY_PROMPT,
)
from shared.utils import (
    print_section_header,
    load_vector_store,
    save_vector_store,
)

# Verify API key
verify_api_key()

print("✓ All imports successful")
print(f"✓ Using model: {DEFAULT_MODEL}")
print(f"✓ Using embeddings: {HF_EMBEDDING_MODEL}")

OK deepseek API key: LOADED
  Preview: sk-0501...20ae
  Base URL: https://api.deepseek.com
✓ All imports successful
✓ Using model: deepseek-v4-flash
✓ Using embeddings: sentence-transformers/all-MiniLM-L6-v2


## 2. 连接到Chinook数据库

In [2]:
print_section_header("Connecting to Chinook Database")

# Path to Chinook database
DB_PATH = PROJECT_ROOT / "data" / "chinook.db"

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Chinook database not found at {DB_PATH}.\n"
        "Please download it from: https://github.com/lerocha/chinook-database"
    )

# Create read-only connection
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
cursor = conn.cursor()

print(f"✓ Connected to database: {DB_PATH}")

# Get table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [row[0] for row in cursor.fetchall()]

print(f"\n✓ Found {len(tables)} tables:")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"  • {table}: {count} rows")


CONNECTING TO CHINOOK DATABASE

✓ Connected to database: D:\ZKS_Data\AI\langchain-rag-tutorial\data\chinook.db

✓ Found 11 tables:
  • Album: 347 rows
  • Artist: 275 rows
  • Customer: 59 rows
  • Employee: 8 rows
  • Genre: 25 rows
  • Invoice: 412 rows
  • InvoiceLine: 2240 rows
  • MediaType: 5 rows
  • Playlist: 18 rows
  • PlaylistTrack: 8715 rows
  • Track: 3503 rows


## 3. 检查数据库Schema

In [3]:
def get_table_schema(conn: sqlite3.Connection, table_name: str) -> str:
    """
    Get the schema for a table.
    
    Args:
        conn: Database connection
        table_name: Name of table
    
    Returns:
        Schema description string
    """
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = cursor.fetchall()
    
    schema_lines = [f"Table: {table_name}"]
    schema_lines.append("Columns:")
    
    for col in columns:
        col_id, name, col_type, not_null, default, pk = col
        constraints = []
        if pk:
            constraints.append("PRIMARY KEY")
        if not_null:
            constraints.append("NOT NULL")
        
        constraint_str = f" ({', '.join(constraints)})" if constraints else ""
        schema_lines.append(f"  - {name}: {col_type}{constraint_str}")
    
    return "\n".join(schema_lines)


print_section_header("Database Schema Inspection")

# Show schema for a few example tables
example_tables = ["Artist", "Album", "Track", "Customer", "Invoice"]

for table in example_tables:
    if table in tables:
        print(f"\n{table}:")
        print("-" * 80)
        print(get_table_schema(conn, table))

print("\n✓ Schema inspection complete")


DATABASE SCHEMA INSPECTION


Artist:
--------------------------------------------------------------------------------
Table: Artist
Columns:
  - ArtistId: INTEGER (PRIMARY KEY, NOT NULL)
  - Name: NVARCHAR(120)

Album:
--------------------------------------------------------------------------------
Table: Album
Columns:
  - AlbumId: INTEGER (PRIMARY KEY, NOT NULL)
  - Title: NVARCHAR(160) (NOT NULL)
  - ArtistId: INTEGER (NOT NULL)

Track:
--------------------------------------------------------------------------------
Table: Track
Columns:
  - TrackId: INTEGER (PRIMARY KEY, NOT NULL)
  - Name: NVARCHAR(200) (NOT NULL)
  - AlbumId: INTEGER
  - MediaTypeId: INTEGER (NOT NULL)
  - GenreId: INTEGER
  - Composer: NVARCHAR(220)
  - Milliseconds: INTEGER (NOT NULL)
  - Bytes: INTEGER
  - UnitPrice: NUMERIC(10,2) (NOT NULL)

Customer:
--------------------------------------------------------------------------------
Table: Customer
Columns:
  - CustomerId: INTEGER (PRIMARY KEY, NOT NULL)
  - F

## 4. 创建Schema嵌入

我们将嵌入表schema以实现相关表的语义检索。

In [4]:
print_section_header("Creating Schema Embeddings")

# Initialize LLM for schema summarization
llm = create_chat_model(
    model=DEFAULT_MODEL,
    temperature=DEFAULT_TEMPERATURE,
)

# Create schema documents with summaries
schema_docs = []

print("\nGenerating semantic descriptions for tables...")

for table in tables:
    schema = get_table_schema(conn, table)
    
    # Generate semantic summary
    summary_chain = SQL_SCHEMA_SUMMARY_PROMPT | llm | StrOutputParser()
    summary = summary_chain.invoke({
        "table_name": table,
        "schema": schema,
    })
    
    # Create document
    doc = Document(
        page_content=f"{table}: {summary}\n\nSchema:\n{schema}",
        metadata={
            "table_name": table,
            "summary": summary,
            "schema": schema,
        },
    )
    schema_docs.append(doc)
    
    print(f"  ✓ {table}")

print(f"\n✓ Created {len(schema_docs)} schema documents")

# Create vector store for schema
embeddings = create_embeddings()
schema_store_path = VECTOR_STORE_DIR / "sql_rag_schema"

schema_vectorstore = load_vector_store(schema_store_path, embeddings)

if schema_vectorstore is None:
    print("\nCreating schema vector store...")
    schema_vectorstore = FAISS.from_documents(schema_docs, embeddings)
    save_vector_store(schema_vectorstore, schema_store_path)
    print("✓ Schema vector store created")
else:
    print("✓ Loaded existing schema vector store")

# Create retriever
schema_retriever = schema_vectorstore.as_retriever(search_kwargs={"k": 3})
print("✓ Schema retriever ready")


CREATING SCHEMA EMBEDDINGS



langchain-openai injected a custom httpx transport to apply `http_socket_options`, which disables httpx's proxy auto-detection (system proxy configuration detected). Set `LANGCHAIN_OPENAI_TCP_KEEPALIVE=0` or pass `http_socket_options=()` to restore default proxy behavior, or supply `openai_proxy` / your own `http_client` / `http_async_client` to take full control.



Generating semantic descriptions for tables...
  ✓ Album
  ✓ Artist
  ✓ Customer
  ✓ Employee
  ✓ Genre
  ✓ Invoice
  ✓ InvoiceLine
  ✓ MediaType
  ✓ Playlist
  ✓ PlaylistTrack
  ✓ Track

✓ Created 11 schema documents


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded vector store from D:\ZKS_Data\AI\langchain-rag-tutorial\data\vector_stores\sql_rag_schema
✓ Loaded existing schema vector store
✓ Schema retriever ready


## 5. 测试Schema检索

In [5]:
print_section_header("Testing Schema Retrieval")

test_queries = [
    "Which tables contain information about songs and music?",
    "Where can I find customer purchase data?",
    "Show me tables related to employees and sales",
]

for query in test_queries:
    print(f"\nQuery: {query}")
    print("-" * 80)
    
    relevant_tables = schema_retriever.invoke(query)
    
    print("Relevant tables:")
    for doc in relevant_tables:
        table_name = doc.metadata["table_name"]
        summary = doc.metadata["summary"]
        print(f"  • {table_name}: {summary}")


TESTING SCHEMA RETRIEVAL


Query: Which tables contain information about songs and music?
--------------------------------------------------------------------------------
Relevant tables:
  • Album: This table stores music album metadata, including a unique album ID, the album title, and a foreign key linking to an artist. It can answer questions such as "What albums are in the database?" or "Which albums belong to a specific artist?"
  • Track: This table stores metadata for individual music tracks, including their title, composer, duration, file size, and unit price. Key columns link each track to its album, genre, and media type (e.g., MPEG, AAC). It can answer questions like “What are the longest tracks in a given album?” or “What is the average price of tracks in a specific genre?”
  • Playlist: This table stores all playlists in a music library, with each row representing a named playlist. The **PlaylistId** uniquely identifies each playlist, while the **Name** column holds its 

## 6. 实现安全的SQL执行

In [6]:
def execute_sql_safely(
    conn: sqlite3.Connection,
    query: str,
    max_results: int = 100,
) -> Tuple[bool, Any, str]:
    """
    Execute SQL query safely with validation.
    
    Args:
        conn: Database connection (should be read-only)
        query: SQL query to execute
        max_results: Maximum rows to return
    
    Returns:
        Tuple of (success, results/error, error_message)
    """
    # Safety checks
    query_upper = query.upper().strip()
    
    # Block dangerous operations
    forbidden = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "CREATE", "TRUNCATE"]
    for keyword in forbidden:
        if keyword in query_upper:
            return False, None, f"Forbidden keyword: {keyword}"
    
    # Must start with SELECT
    if not query_upper.startswith("SELECT"):
        return False, None, "Only SELECT queries are allowed"
    
    try:
        cursor = conn.cursor()
        cursor.execute(query)
        
        # Fetch results with limit
        results = cursor.fetchmany(max_results)
        
        # Get column names
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        
        # Convert to list of dicts
        results_list = [
            dict(zip(columns, row))
            for row in results
        ]
        
        return True, results_list, ""
    
    except Exception as e:
        return False, None, str(e)


print("✓ Safe SQL execution function defined")

# Test
print("\nTest query: SELECT * FROM Artist LIMIT 3")
success, results, error = execute_sql_safely(conn, "SELECT * FROM Artist LIMIT 3")

if success:
    print("\n✓ Query successful!")
    df = pd.DataFrame(results)
    print(df)
else:
    print(f"\n❌ Query failed: {error}")

✓ Safe SQL execution function defined

Test query: SELECT * FROM Artist LIMIT 3

✓ Query successful!
   ArtistId       Name
0         1      AC/DC
1         2     Accept
2         3  Aerosmith


## 7. 构建Text-to-SQL管道

In [7]:
def text_to_sql_rag(
    question: str,
    conn: sqlite3.Connection,
    schema_retriever,
    llm,
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Complete Text-to-SQL RAG pipeline.
    
    Args:
        question: Natural language question
        conn: Database connection
        schema_retriever: Schema retriever
        llm: Language model
        verbose: Print debug info
    
    Returns:
        Dict with query, results, answer, etc.
    """
    if verbose:
        print(f"\n[SQL RAG] Question: {question}")
    
    # 1. Retrieve relevant schema
    relevant_schemas = schema_retriever.invoke(question)
    
    schema_context = "\n\n".join([
        doc.metadata["schema"]
        for doc in relevant_schemas
    ])
    
    if verbose:
        tables = [doc.metadata["table_name"] for doc in relevant_schemas]
        print(f"[SQL RAG] Relevant tables: {', '.join(tables)}")
    
    # 2. Generate SQL query
    sql_chain = TEXT_TO_SQL_PROMPT | llm | StrOutputParser()
    sql_query = sql_chain.invoke({
        "schema": schema_context,
        "question": question,
    }).strip()
    
    # Clean SQL (remove markdown formatting if present)
    sql_query = sql_query.replace("``sql", "").replace("```", "").strip()
    
    if verbose:
        print(f"[SQL RAG] Generated SQL: {sql_query}")
    
    # 3. Execute SQL
    success, results, error = execute_sql_safely(conn, sql_query)
    
    if not success:
        if verbose:
            print(f"[SQL RAG] Query failed: {error}")
            print("[SQL RAG] Attempting error recovery...")
        
        # Try error recovery
        recovery_chain = SQL_ERROR_RECOVERY_PROMPT | llm | StrOutputParser()
        corrected_sql = recovery_chain.invoke({
            "question": question,
            "failed_query": sql_query,
            "error": error,
            "schema": schema_context,
        }).strip().replace("```sql", "").replace("```", "").strip()
        
        if verbose:
            print(f"[SQL RAG] Corrected SQL: {corrected_sql}")
        
        success, results, error = execute_sql_safely(conn, corrected_sql)
        
        if success:
            sql_query = corrected_sql
        else:
            return {
                "question": question,
                "sql_query": sql_query,
                "success": False,
                "error": error,
                "answer": f"I couldn't generate a valid SQL query. Error: {error}",
            }
    
    if verbose:
        print(f"[SQL RAG] Query successful! Rows returned: {len(results)}")
    
    # 4. Interpret results
    results_str = json.dumps(results, indent=2) if results else "No results found"
    
    interpret_chain = SQL_RESULTS_INTERPRETATION_PROMPT | llm | StrOutputParser()
    answer = interpret_chain.invoke({
        "question": question,
        "sql_query": sql_query,
        "results": results_str,
    })
    
    return {
        "question": question,
        "sql_query": sql_query,
        "success": True,
        "results": results,
        "answer": answer,
    }


print("✓ Text-to-SQL RAG pipeline defined")

✓ Text-to-SQL RAG pipeline defined


## 8. 用各种查询测试SQL RAG

In [8]:
print_section_header("Testing SQL RAG")

test_questions = [
    "How many customers are in the database?",
    "What are the top 5 longest songs?",
    "Show me the total sales by country, ordered by highest revenue",
    "Which artist has released the most albums?",
    "What's the average track length in minutes?",
]

for i, question in enumerate(test_questions, 1):
    print("\n" + "=" * 80)
    print(f"Question {i}: {question}")
    print("=" * 80)
    
    result = text_to_sql_rag(
        question=question,
        conn=conn,
        schema_retriever=schema_retriever,
        llm=llm,
        verbose=True,
    )
    
    print("\n" + "-" * 80)
    print("ANSWER:")
    print("-" * 80)
    print(result["answer"])
    
    # Show results as DataFrame if available
    if result["success"] and result["results"]:
        print("\n" + "-" * 80)
        print("DATA:")
        print("-" * 80)
        df = pd.DataFrame(result["results"]).head(10)
        print(df)


TESTING SQL RAG


Question 1: How many customers are in the database?

[SQL RAG] Question: How many customers are in the database?
[SQL RAG] Relevant tables: Customer, Invoice, Artist
[SQL RAG] Generated SQL: SELECT COUNT(*) FROM Customer;
[SQL RAG] Query successful! Rows returned: 1

--------------------------------------------------------------------------------
ANSWER:
--------------------------------------------------------------------------------
There are 59 customers in the database.

--------------------------------------------------------------------------------
DATA:
--------------------------------------------------------------------------------
   COUNT(*)
0        59

Question 2: What are the top 5 longest songs?

[SQL RAG] Question: What are the top 5 longest songs?
[SQL RAG] Relevant tables: Track, Album, Genre
[SQL RAG] Generated SQL: SELECT Name, Milliseconds
FROM Track
ORDER BY Milliseconds DESC
LIMIT 5;
[SQL RAG] Query successful! Rows returned: 5

-----------------

## 9. 带JOIN的复杂查询

In [9]:
print_section_header("Testing Complex Queries with JOINs")

complex_questions = [
    "Show me the top 3 customers by total amount spent",
    "Which genres have the most tracks?",
    "List all employees and how many customers they support",
]

for i, question in enumerate(complex_questions, 1):
    print("\n" + "=" * 80)
    print(f"Complex Query {i}: {question}")
    print("=" * 80)
    
    result = text_to_sql_rag(
        question=question,
        conn=conn,
        schema_retriever=schema_retriever,
        llm=llm,
        verbose=True,
    )
    
    print("\n" + "-" * 80)
    print("ANSWER:")
    print("-" * 80)
    print(result["answer"])
    
    if result["success"] and result["results"]:
        print("\n" + "-" * 80)
        print("DATA:")
        print("-" * 80)
        df = pd.DataFrame(result["results"])
        print(df)


TESTING COMPLEX QUERIES WITH JOINS


Complex Query 1: Show me the top 3 customers by total amount spent

[SQL RAG] Question: Show me the top 3 customers by total amount spent
[SQL RAG] Relevant tables: Invoice, InvoiceLine, Customer
[SQL RAG] Generated SQL: SELECT c.CustomerId, c.FirstName, c.LastName, SUM(i.Total) AS TotalSpent
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId, c.FirstName, c.LastName
ORDER BY TotalSpent DESC
LIMIT 3;
[SQL RAG] Query successful! Rows returned: 3

--------------------------------------------------------------------------------
ANSWER:
--------------------------------------------------------------------------------
Here are the top 3 customers by total amount spent:

1. **Helena Holý** – $49.62  
2. **Richard Cunningham** – $47.62  
3. **Luis Rojas** – $46.62  

These three customers had the highest total spending based on the data.

--------------------------------------------------------------------------------
DATA:

## 10. 性能指标

In [10]:
import time

print_section_header("Performance Metrics")

# Benchmark query
benchmark_query = "How many albums are there in total?"

print(f"Benchmark query: {benchmark_query}\n")

start = time.time()
result = text_to_sql_rag(
    question=benchmark_query,
    conn=conn,
    schema_retriever=schema_retriever,
    llm=llm,
    verbose=False,
)
elapsed = time.time() - start

print("=" * 80)
print("PERFORMANCE BREAKDOWN:")
print("=" * 80)
print(f"Total time: {elapsed:.2f}s")
print("\nComponents:")
print("  • Schema retrieval: ~0.5-1.0s (vector search)")
print("  • SQL generation: ~1.0-2.0s (LLM call)")
print("  • SQL execution: <0.1s (fast on indexed DB)")
print("  • Results interpretation: ~1.0-2.0s (LLM call)")

print("\n" + "=" * 80)
print("COST ANALYSIS:")
print("=" * 80)
print("LLM Calls per query:")
print("  • Schema summarization: 1 (one-time, cached)")
print("  • SQL generation: 1")
print("  • Error recovery: 0-1 (if needed)")
print("  • Results interpretation: 1")
print("  • Total: 2-3 LLM calls")
print("\nVector searches: 1 (schema retrieval)")
print("Database queries: 1 (SQL execution)")


PERFORMANCE METRICS

Benchmark query: How many albums are there in total?

PERFORMANCE BREAKDOWN:
Total time: 2.29s

Components:
  • Schema retrieval: ~0.5-1.0s (vector search)
  • SQL generation: ~1.0-2.0s (LLM call)
  • SQL execution: <0.1s (fast on indexed DB)
  • Results interpretation: ~1.0-2.0s (LLM call)

COST ANALYSIS:
LLM Calls per query:
  • Schema summarization: 1 (one-time, cached)
  • SQL generation: 1
  • Error recovery: 0-1 (if needed)
  • Results interpretation: 1
  • Total: 2-3 LLM calls

Vector searches: 1 (schema retrieval)
Database queries: 1 (SQL execution)


## 11. 混合方法: SQL + Vector RAG

为了获得最大的灵活性,将SQL RAG与传统vector RAG结合使用。

In [11]:
print_section_header("Hybrid SQL + Vector RAG")

print("\nThis approach routes queries to either:")
print("  • SQL RAG: For structured queries (counts, aggregations, filters)")
print("  • Vector RAG: For semantic queries (document search, similarity)")

# Simple query classifier
def classify_query(question: str) -> str:
    """
    Classify if query needs SQL or vector RAG.
    """
    sql_keywords = [
        "how many", "count", "total", "sum", "average", "maximum", "minimum",
        "top", "bottom", "highest", "lowest", "most", "least",
        "by country", "by genre", "by artist", "per",
    ]
    
    question_lower = question.lower()
    
    for keyword in sql_keywords:
        if keyword in question_lower:
            return "SQL"
    
    return "VECTOR"


# Test classification
test_cases = [
    "How many customers are in France?",  # SQL
    "What is LCEL in LangChain?",  # VECTOR
    "Show me the top 5 albums",  # SQL
    "Explain the concept of retrieval",  # VECTOR
]

print("\n" + "=" * 80)
print("Query Classification Examples:")
print("=" * 80)

for query in test_cases:
    classification = classify_query(query)
    print(f"\n'{query}'")
    print(f"  → {classification} RAG")

print("\n" + "=" * 80)
print("Implementation Note:")
print("=" * 80)
print("A production system would:")
print("  1. Use LLM to classify query intent")
print("  2. Route to appropriate RAG system")
print("  3. Fall back to alternative if primary fails")
print("  4. Combine results if needed")


HYBRID SQL + VECTOR RAG


This approach routes queries to either:
  • SQL RAG: For structured queries (counts, aggregations, filters)
  • Vector RAG: For semantic queries (document search, similarity)

Query Classification Examples:

'How many customers are in France?'
  → SQL RAG

'What is LCEL in LangChain?'
  → VECTOR RAG

'Show me the top 5 albums'
  → SQL RAG

'Explain the concept of retrieval'
  → VECTOR RAG

Implementation Note:
A production system would:
  1. Use LLM to classify query intent
  2. Route to appropriate RAG system
  3. Fall back to alternative if primary fails
  4. Combine results if needed


## 12. 关键要点

### 总结

**SQL RAG** 实现对结构化数据库的自然语言查询:
- Schema检索找到相关表
- Text-to-SQL使用LLM生成查询
- 安全执行防止危险操作
- 结果解释提供自然回答
- 错误恢复处理SQL生成失败

### 管道组件

1. **Schema索引**(一次性): 嵌入表描述
2. **Schema检索**: 找到相关表(vector搜索)
3. **SQL生成**: LLM使用schema上下文创建查询
4. **验证**: 安全检查(只读,无注入)
5. **执行**: 在受控环境中运行查询
6. **解释**: 将结果转换为自然语言
7. **错误恢复**: 如果需要,重试并修正

### 最佳实践

**Schema设计:**
- ✅ 使用描述性的表/列名
- ✅ 添加注释和文档
- ✅ 维护引用完整性
- ✅ 为表创建语义摘要

**安全性:**
- ✅ 始终使用只读连接
- ✅ 白名单允许的操作(仅SELECT)
- ✅ 设置查询超时
- ✅ 限制结果集大小
- ✅ 验证生成的SQL

**错误处理:**
- ✅ 实现重试逻辑
- ✅ 提供有用的错误消息
- ✅ 记录失败的查询以进行分析
- ✅ 回退到替代方法

**优化:**
- ✅ 缓存schema嵌入
- ✅ 正确索引数据库
- ✅ 使用查询结果缓存
- ✅ 批量处理相似查询

### 局限性

❌ **SQL生成挑战:**
- 复杂的JOIN可能会失败
- 模糊的问题导致错误的查询
- LLM可能会编造表/列名
- 窗口函数和CTE很难处理

❌ **Schema依赖:**
- 需要精心设计的schema
- 命名不当会导致查询质量差
- Schema更改会破坏查询

### 何时使用

选择**SQL RAG**当:
- ✅ 数据在数据库中结构化
- ✅ 需要聚合和分析
- ✅ 精度至关重要
- ✅ 用户询问"多少"、"总计"、"平均"
- ✅ 现有数据库基础设施

选择**Vector RAG**当:
- ✅ 非结构化文本文档
- ✅ 语义相似度很重要
- ✅ 需要模糊匹配
- ✅ 没有清晰的schema

选择**混合**当:
- ✅ 结构化和非结构化数据都有
- ✅ 多样的查询类型
- ✅ 需要最大的灵活性

### 扩展

**高级功能:**
- 多数据库支持(PostgreSQL、MySQL等)
- 查询优化提示
- 缓存查询计划
- 自然语言查询建议
- 查询历史和学习

**生产增强:**
- 查询审批工作流
- 结果解释(EXPLAIN)
- 性能监控
- A/B测试提示
- 用户反馈循环

---

**复杂度评级:** ⭐⭐⭐⭐ (高 - 需要数据库知识 + LLM集成)

**生产就绪性:** ⭐⭐⭐ (中等 - 需要广泛测试和安全措施)

继续学习**15_graphrag.ipynb**了解基于图的知识检索!

## 清理

In [12]:
# Close database connection
conn.close()
print("✓ Database connection closed")

✓ Database connection closed
